# SnakyNet Massive 13x13 Self-Play Training on Colab Pro (VERSION 3 - HEURISTIC AUGMENTED)
This notebook trains the 13x13 AlphaGo-style ResNet for the Snakey game using Google Colab A100 GPUs.
**Updates in V3:** Heuristic Priors, Heuristic Rollouts, Supervised Bootstrapping, and Loss History tracking.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/SnakyNet_Checkpoints', exist_ok=True)

In [ ]:
!rm -rf /content/funSearch2-main
!rm -rf /content/main.zip
!wget -q https://github.com/aanderson3456/funSearch2/archive/refs/heads/main.zip
!unzip -q main.zip
import sys
sys.path.append('/content/funSearch2-main/big_nn')

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import os, glob, math
from env import SnakyEnv
from resnet import SnakyNet
import json

def heuristic_priors(env, legal_moves):
    priors = np.ones(len(legal_moves), dtype=np.float32)
    for i, move in enumerate(legal_moves):
        y, x = divmod(move, env.size)
        if env.current_player == 1:
            score = 1.0
            for dy, dx in [(-1,0), (1,0), (0,-1), (0,1)]:
                ny, nx = y + dy, x + dx
                if 0 <= ny < env.size and 0 <= nx < env.size:
                    if env.maker_board & (1 << (ny * env.size + nx)):
                        score += 5.0
            priors[i] = score
        else:
            if ((x // 2) + (y // 2)) % 2 == 0:
                priors[i] = 10.0
            else:
                priors[i] = 1.0
    return priors

def heuristic_rollout(env_state, max_depth=10):
    env = clone_env(env_state)
    depth = 0
    while not env.done and depth < max_depth:
        legal_moves = env.get_legal_moves()
        priors = heuristic_priors(env, legal_moves)
        priors /= np.sum(priors)
        best_move = legal_moves[np.argmax(priors)]
        env.step(best_move)
        depth += 1
    if env.done:
        return 1.0 if env.winner == 1 else -1.0
    return 0.0

class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.buffer = []
        
    def add(self, data):
        self.buffer.extend(data)
        if len(self.buffer) > self.capacity:
            self.buffer = self.buffer[-self.capacity:]
            
    def sample(self, batch_size):
        idx = np.random.choice(len(self.buffer), batch_size, replace=False)
        return [self.buffer[i] for i in idx]

class Node:
    def __init__(self, parent=None, prior_prob=1.0):
        self.parent = parent
        self.children = {}
        self.visit_count = 0
        self.value_sum = 0
        self.prior_prob = prior_prob

    def is_expanded(self):
        return len(self.children) > 0

    def get_value(self):
        if self.visit_count == 0:
            return 0
        return self.value_sum / self.visit_count

def _encode_state_batch(envs, device):
    states = np.zeros((len(envs), 3, envs[0].size, envs[0].size), dtype=np.float32)
    for idx, env in enumerate(envs):
        for i in range(env.size * env.size):
            y, x = divmod(i, env.size)
            if env.maker_board & (1 << i):
                states[idx, 0, y, x] = 1.0
            if env.breaker_board & (1 << i):
                states[idx, 1, y, x] = 1.0
        if env.current_player == 1:
            states[idx, 2, :, :] = 1.0
        else:
            states[idx, 2, :, :] = 0.0
    return torch.tensor(states, dtype=torch.float32, device=device)

def clone_env(env):
    new_env = SnakyEnv(size=env.size)
    new_env.maker_board = env.maker_board
    new_env.breaker_board = env.breaker_board
    new_env.current_player = env.current_player
    new_env.done = env.done
    new_env.winner = env.winner
    new_env.win_masks = env.win_masks  # Share reference, zero-copy!
    return new_env

def batched_search(model, envs, num_searches, c_puct=1.0, add_noise=False, device='cuda'):
    roots = [Node() for _ in range(len(envs))]
    
    states_tensor = _encode_state_batch(envs, device)
    with torch.no_grad():
        policy_logits, _ = model(states_tensor)
        policies = F.softmax(policy_logits, dim=1).cpu().numpy()
        
    for i, env in enumerate(envs):
        legal_moves = env.get_legal_moves()
        policy = policies[i]
        valid_policy = np.zeros_like(policy)
        valid_policy[legal_moves] = policy[legal_moves]
        policy_sum = np.sum(valid_policy)
        if policy_sum > 0:
            valid_policy /= policy_sum
        else:
            valid_policy[legal_moves] = 1.0 / len(legal_moves)
            
        h_priors = heuristic_priors(env, legal_moves)
        h_priors /= np.sum(h_priors)
        valid_policy_blended = 0.5 * valid_policy[legal_moves] + 0.5 * h_priors
        for m_idx, move in enumerate(legal_moves):
            roots[i].children[move] = Node(parent=roots[i], prior_prob=valid_policy_blended[m_idx])
            
        if add_noise:
            dirichlet_alpha = 0.3
            dirichlet_noise = np.random.dirichlet([dirichlet_alpha] * len(legal_moves))
            frac = 0.25
            for idx_m, move in enumerate(legal_moves):
                roots[i].children[move].prior_prob = (1 - frac) * roots[i].children[move].prior_prob + frac * dirichlet_noise[idx_m]

    for _ in range(num_searches):
        search_envs = [clone_env(env) for env in envs]
        search_nodes = [root for root in roots]
        
        for i in range(len(envs)):
            node = search_nodes[i]
            env = search_envs[i]
            while node.is_expanded():
                best_ucb = -float('inf')
                best_action = None
                best_child = None
                
                for action, child in node.children.items():
                    q_val = child.get_value()
                    q = q_val if env.current_player == 1 else -q_val
                    u = c_puct * child.prior_prob * math.sqrt(node.visit_count) / (1 + child.visit_count)
                    ucb = q + u
                    
                    if ucb > best_ucb:
                        best_ucb = ucb
                        best_action = action
                        best_child = child
                        
                node = best_child
                env.step(best_action)
            search_nodes[i] = node
            
        states_to_eval = []
        eval_indices = []
        for i in range(len(envs)):
            if not search_envs[i].done:
                states_to_eval.append(search_envs[i])
                eval_indices.append(i)
            else:
                v = 1.0 if search_envs[i].winner == 1 else -1.0
                node = search_nodes[i]
                while node is not None:
                    node.visit_count += 1
                    node.value_sum += v
                    node = node.parent
                    
        if states_to_eval:
            states_tensor = _encode_state_batch(states_to_eval, device)
            with torch.no_grad():
                policy_logits, values = model(states_tensor)
                policies = F.softmax(policy_logits, dim=1).cpu().numpy()
                values = values.cpu().numpy()
                
            for idx, i in enumerate(eval_indices):
                node = search_nodes[i]
                env = search_envs[i]
                policy = policies[idx]
                v = values[idx][0]
                
                legal_moves = env.get_legal_moves()
                valid_policy = np.zeros_like(policy)
                valid_policy[legal_moves] = policy[legal_moves]
                policy_sum = np.sum(valid_policy)
                if policy_sum > 0:
                    valid_policy /= policy_sum
                else:
                    valid_policy[legal_moves] = 1.0 / len(legal_moves)
                    
                h_priors = heuristic_priors(env, legal_moves)
                h_priors /= np.sum(h_priors)
                valid_policy_blended = 0.5 * valid_policy[legal_moves] + 0.5 * h_priors
                for m_idx, move in enumerate(legal_moves):
                    node.children[move] = Node(parent=node, prior_prob=valid_policy_blended[m_idx])
                    
                rollout_v = heuristic_rollout(env, max_depth=10)
                blended_v = 0.5 * v + 0.5 * rollout_v
                
                while node is not None:
                    node.visit_count += 1
                    node.value_sum += blended_v
                    node = node.parent
                    
    action_probs_batch = []
    for i in range(len(envs)):
        action_probs = np.zeros(envs[i].size * envs[i].size)
        for action, child in roots[i].children.items():
            action_probs[action] = child.visit_count / num_searches
        action_probs_batch.append(action_probs)
        
    return action_probs_batch

def self_play(model, num_games=100, mcts_searches=100, device='cuda'):
    envs = [SnakyEnv(size=13) for _ in range(num_games)]
    all_data = []
    
    active_indices = list(range(num_games))
    game_data = [[] for _ in range(num_games)]
    games_finished = 0
    
    move_count = 0
    while active_indices:
        move_count += 1
        print(f"\rProcessing move {move_count} simultaneously for {len(active_indices)} active games...", end='')
        active_envs = [envs[i] for i in active_indices]
        action_probs_batch = batched_search(model, active_envs, mcts_searches, add_noise=True, device=device)
        
        next_active = []
        for idx, env_idx in enumerate(active_indices):
            env = envs[env_idx]
            action_probs = action_probs_batch[idx]
            game_data[env_idx].append((env.maker_board, env.breaker_board, env.current_player, action_probs))
            
            if len(game_data[env_idx]) < 15:
                action = np.random.choice(len(action_probs), p=action_probs)
            else:
                action = np.argmax(action_probs)
                
            env.step(action)
            
            if not env.done:
                next_active.append(env_idx)
            else:
                v = 1.0 if env.winner == 1 else -1.0
                augmented_data = []
                for mb, bb, cp, policy in game_data[env_idx]:
                    symmetries = env.get_symmetries(mb, bb, policy)
                    for s_mb, s_bb, s_policy in symmetries:
                        augmented_data.append((s_mb, s_bb, cp, s_policy, v))
                all_data.extend(augmented_data)
                games_finished += 1
                
        active_indices = next_active
        
    print(f"\nFinished all {num_games} games!")
    return all_data

def generate_heuristic_games(num_games):
    print(f"Generating {num_games} heuristic games for supervised bootstrapping...")
    all_data = []
    for _ in range(num_games):
        env = SnakyEnv(size=13)
        game_data = []
        while not env.done:
            legal_moves = env.get_legal_moves()
            priors = heuristic_priors(env, legal_moves)
            priors /= np.sum(priors)
            action_probs = np.zeros(env.size * env.size)
            action_probs[legal_moves] = priors
            game_data.append((env.maker_board, env.breaker_board, env.current_player, action_probs))
            action = np.random.choice(legal_moves, p=priors)
            env.step(action)
        v = 1.0 if env.winner == 1 else -1.0
        augmented_data = []
        for mb, bb, cp, policy in game_data:
            symmetries = env.get_symmetries(mb, bb, policy)
            for s_mb, s_bb, s_policy in symmetries:
                augmented_data.append((s_mb, s_bb, cp, s_policy, v))
        all_data.extend(augmented_data)
    print(f"Generated {len(all_data)} states from heuristic play!")
    return all_data

def train(model, buffer, batch_size=256, epochs=5, lr=0.001, device='cuda'):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    model.train()
    total_loss = 0
    
    for epoch in range(epochs):
        data = buffer.sample(batch_size)
        states = np.zeros((batch_size, 3, 13, 13), dtype=np.float32)
        target_policies = np.zeros((batch_size, 169), dtype=np.float32)
        target_values = np.zeros((batch_size, 1), dtype=np.float32)
        
        for i, (mb, bb, cp, p, v) in enumerate(data):
            for bit in range(169):
                y, x = divmod(bit, 13)
                if mb & (1 << bit):
                    states[i, 0, y, x] = 1.0
                if bb & (1 << bit):
                    states[i, 1, y, x] = 1.0
            states[i, 2, :, :] = 1.0 if cp == 1 else 0.0
            target_policies[i] = p
            target_values[i] = v
            
        states = torch.tensor(states, dtype=torch.float32, device=device)
        target_policies = torch.tensor(target_policies, dtype=torch.float32, device=device)
        target_values = torch.tensor(target_values, dtype=torch.float32, device=device)
        
        optimizer.zero_grad()
        out_policy, out_value = model(states)
        
        log_probs = F.log_softmax(out_policy, dim=1)
        policy_loss = -(target_policies * log_probs).sum(dim=1).mean()
        value_loss = F.mse_loss(out_value, target_values)
        
        loss = policy_loss + value_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / epochs

In [ ]:
# Force CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')

model = SnakyNet(num_resBlocks=16, num_channels=256, board_size=13).to(device)
buffer = ReplayBuffer(capacity=100000)

import re
drive_models = glob.glob("/content/drive/MyDrive/SnakyNet_Checkpoints/snaky_large_model_it*.pt")
root_models = glob.glob("/content/drive/MyDrive/snaky_large_model_it*.pt")
all_models = drive_models + root_models

start_iteration = 13
if all_models:
    def get_it(path):
        m = re.search(r'it(\d+)\.pt', path)
        return int(m.group(1)) if m else -1
    latest_model_path = max(all_models, key=get_it)
    start_iteration = get_it(latest_model_path) + 1
    print(f"\nFound checkpoint: {latest_model_path}")
    loaded = torch.load(latest_model_path, weights_only=False, map_location=device)
    if isinstance(loaded, dict):
        model.load_state_dict(loaded)
        print("Successfully loaded weights from state_dict!")
    else:
        model.load_state_dict(loaded.state_dict())
        print("Successfully extracted and loaded weights from full model!")
else:
    print("\nNo checkpoints found. Starting from scratch!")

loss_history = []
loss_path = '/content/drive/MyDrive/SnakyNet_Checkpoints/loss_history.json'
if os.path.exists(loss_path):
    with open(loss_path, 'r') as f:
        loss_history = json.load(f)

if start_iteration == 0:
    print('Performing supervised bootstrapping...')
    bs_data = generate_heuristic_games(500)
    buffer.add(bs_data)
    loss = train(model, buffer, batch_size=batch_size, epochs=20, device=device)
    print(f'Bootstrap Loss: {loss:.4f}')

iterations = 1000
games_per_iter = 100  # A100 is blazing fast
batch_size = 512      # A100 has huge VRAM

for it in range(start_iteration, iterations):
    print(f'\n--- Iteration {it}/{iterations} ---')
    model.eval()
    print('Starting Self-Play...')
    data = self_play(model, num_games=games_per_iter, mcts_searches=100, device=device)
    buffer.add(data)
    
    if len(buffer.buffer) >= batch_size: 
        loss = train(model, buffer, batch_size=batch_size, epochs=10, device=device)
        print(f'Training Loss: {loss:.4f}')
        loss_history.append({'iteration': it, 'loss': loss})
        with open(loss_path, 'w') as f:
            json.dump(loss_history, f)
        
    # Always save properly as state_dict so it loads anywhere!
    save_path = f'/content/drive/MyDrive/SnakyNet_Checkpoints/snaky_large_model_it{it}.pt'
    torch.save(model.state_dict(), save_path)
    print(f'Saved Checkpoint to Google Drive! ({save_path})')
